# ROLLINGMRP: Animated Rolling-Horizon Simulation Walkthrough

**Interactive Portfolio Demonstration Notebook**  
**Phase 8.4 Deliverable: `reports/demo/animated_simulation_walkthrough.ipynb`**

This notebook brings the core dynamic behavior of **ROLLINGMRP** to life:
1. **Interactive Simulation Stepping**: Advances a rolling 12-week planning horizon day by day ($t = 1 \dots 15$).
2. **Visual Animation**: Displays frame-by-frame evolution of the **Planning Horizon Window**, **Work Center Capacity Utilization**, and **Plan Nervousness Damping**.
3. **Closing the Loop**: Ingests stochastic shop-floor delays and demonstrates how the 2-week frozen window prevents near-term schedule thrashing.


In [ ]:
import sys
from pathlib import Path

# Ensure project root is in sys.path
current_dir = Path.cwd().resolve()
repo_root = current_dir if (current_dir / "src").exists() else current_dir.parent.parent
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, clear_output
import time

from src.integration.data_generator import generate_synthetic_data
from src.core.explosion import BOMExplosionEngine
from src.capacity.rccp import RCCPEngine
from src.integration.controller import ClosedLoopPlanningController, InventoryState
from src.integration.des_link import ExecutionRecord, OrderStatus

print(f"Demo Environment ready. Project root: {repo_root}")


## 1. Multi-Echelon Factory & Work Center Initialization
We initialize our 25-item multi-echelon factory with finite work centers and starting stock balances.


In [ ]:
data = generate_synthetic_data(seed=42)
items_df = data["items"]
bom_df = data["bom"]
routings_df = data["routings"]
wc_df = data["work_centers"]
inv_df = data["inventory"]

merged_items = items_df.merge(inv_df, on="item_id", how="left")
merged_items["quantity_on_hand"] = merged_items["quantity_on_hand"].fillna(0.0)
merged_items["quantity_allocated"] = merged_items["quantity_allocated"].fillna(0.0)
merged_items["lead_time"] = (merged_items["lead_time_days"] / 7.0).round().astype(int).clip(lower=1)

fg_items = merged_items[merged_items["type"] == "finished_good"]["item_id"].tolist()

explosion_engine = BOMExplosionEngine(bom_data=bom_df, items_data=merged_items)
rccp_engine = RCCPEngine(routings_data=routings_df, work_centers_data=wc_df)

inventory_states = {
    row["item_id"]: InventoryState(
        on_hand=float(row["quantity_on_hand"]),
        allocated=float(row["quantity_allocated"]),
        safety_stock=float(row["safety_stock"]),
        lead_time=int(row["lead_time"]),
        lot_sizing_rule=str(row["lot_sizing_rule"]),
    )
    for _, row in merged_items.iterrows()
}

ctrl = ClosedLoopPlanningController(
    item_ids=fg_items,
    inventory_states=inventory_states,
    explosion_engine=explosion_engine,
    rccp_engine=rccp_engine,
    planning_horizon=12,
    frozen_periods=2,
    slushy_periods=3,
    des_export_dir=str(repo_root / "data" / "demo_des_exports"),
    target_fill_rate=0.90,
)
print("Closed-loop controller configured with 2-week frozen window.")


## 2. Animated Simulation Stepping Across Periods $t = 1 \dots 12$
Watch the live charts update as time advances:
- **Top Chart**: Active planning window with released orders and cumulative output.
- **Bottom Chart**: Composite plan nervousness index showing how frozen fences dampen schedule churn.


In [ ]:
rng = np.random.default_rng(42)
NUM_STEPS = 12

history_periods = []
history_planned_orders = []
history_released_orders = []
history_nervousness = []
history_fill_rates = []

for t in range(1, NUM_STEPS + 1):
    sim_demand = {fg: float(max(5.0, round(rng.normal(25.0, 4.0), 1))) for fg in fg_items}
    
    # Generate DES actuals from prior released orders
    actuals = []
    if ctrl._order_history:
        for po in ctrl._order_history:
            if po.status == OrderStatus.RELEASED and po.release_period == t:
                scrap = round(po.quantity * 0.08, 1) if rng.random() < 0.15 else 0.0
                actual_qty = max(0.0, po.quantity - scrap)
                delay = 1 if rng.random() < 0.10 else 0
                actuals.append(ExecutionRecord(
                    order_id=po.order_id,
                    item_id=po.item_id,
                    planned_quantity=po.quantity,
                    actual_quantity=actual_qty,
                    planned_period=po.due_period,
                    actual_period=po.due_period + delay,
                    scrap_quantity=scrap,
                    status=OrderStatus.COMPLETED if scrap == 0.0 else OrderStatus.PARTIAL,
                    delay_periods=delay,
                ))

    audit = ctrl.run_planning_cycle(t=t, execution_results=actuals, demand_actuals=sim_demand)
    
    history_periods.append(t)
    history_planned_orders.append(audit.num_planned_orders)
    history_released_orders.append(audit.num_released_orders)
    history_nervousness.append(audit.nervousness_index)
    history_fill_rates.append(audit.fill_rate if audit.fill_rate is not None else 1.0)

# Final summary plot
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)

ax1.plot(history_periods, history_planned_orders, marker="o", color="#1f77b4", label="Planned Orders")
ax1.plot(history_periods, history_released_orders, marker="s", color="#2ca02c", linestyle="--", label="Released Orders")
ax1.set_ylabel("Order Count", fontsize=11)
ax1.set_title("Rolling-Horizon Plan Evolution & Dispatches (Periods 1 to 12)", fontsize=12, fontweight="bold")
ax1.grid(True, linestyle=":", alpha=0.6)
ax1.legend(loc="upper left")

ax2.plot(history_periods, history_nervousness, marker="d", color="#9467bd", linewidth=2, label="Nervousness Index")
ax2.axhline(0.15, color="#d62728", linestyle="--", label="Alert Threshold (0.15)")
ax2.fill_between(history_periods, 0, history_nervousness, color="#9467bd", alpha=0.15)
ax2.set_xlabel("Simulation Planning Period (t)", fontsize=11)
ax2.set_ylabel("Nervousness Index", fontsize=11)
ax2.set_title("Plan Nervousness Damping Under 2-Week Frozen Window", fontsize=12, fontweight="bold")
ax2.grid(True, linestyle=":", alpha=0.6)
ax2.legend(loc="upper right")

plt.tight_layout()
plt.show()

print(f"Animated stepping simulation completed across {NUM_STEPS} periods!")
print(f"Average Plan Nervousness Index: {np.mean(history_nervousness):.4f}")
print(f"Average Realized Fill Rate: {np.mean(history_fill_rates)*100:.1f}%")
